In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import pickle
import os

In [2]:
df_model = pd.read_csv("/home/vlad/eQTL/processed/df_model.csv")
df_orig = pd.read_csv("/home/vlad/eQTL/processed/classification_with_orf.tsv", sep="\t")

FASTA_PATH = "/home/vlad/eQTL/sqanti3_output/wtc11_corrected.fasta"
PICKLE_PATH = "/home/vlad/eQTL/processed/sequences.pkl"

if os.path.exists(PICKLE_PATH):
    with open(PICKLE_PATH, "rb") as f:
        seqs = pickle.load(f)
else:
    valid_ids = set(df_orig['isoform'])
    seqs = {}
    current_id = None
    current_seq = []
    with open(FASTA_PATH) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id and current_id in valid_ids:
                    seqs[current_id] = ''.join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line.upper())
        if current_id and current_id in valid_ids:
            seqs[current_id] = ''.join(current_seq)
    with open(PICKLE_PATH, "wb") as f:
        pickle.dump(seqs, f)

df = df_orig[['isoform']].copy()
df['sequence'] = df['isoform'].map(seqs)
df['target'] = df_model['target_4class'].values
df = df.dropna(subset=['sequence'])

lengths = df['sequence'].str.len()
print(f"Dataset: {len(df):,} isoforms")
print(f"Sequence length: min={lengths.min()}, median={lengths.median():.0f}, max={lengths.max()}")
print(f"\nClass distribution:")
print(df['target'].value_counts())

Dataset: 489,545 isoforms
Sequence length: min=1, median=2385, max=205012

Class distribution:
target
real_high    202421
uncertain    175559
artifact      66905
real_mid      44660
Name: count, dtype: int64


To one-hot code the DNA, we must check some info:

In [3]:
for max_len in [1000, 2000, 4000, 8000]:
    pct = (lengths <= max_len).mean() * 100
    print(f"MAX_LEN={max_len}: covers {pct:.1f}% of isoforms")

MAX_LEN=1000: covers 26.6% of isoforms
MAX_LEN=2000: covers 42.9% of isoforms
MAX_LEN=4000: covers 75.7% of isoforms
MAX_LEN=8000: covers 98.0% of isoforms


In [4]:
for cls in df['target'].unique():
    cls_lengths = df[df['target'] == cls]['sequence'].str.len()
    pct_over = (cls_lengths > 8000).mean() * 100
    print(f"{cls:12s}: median={cls_lengths.median():.0f}, "
          f"cutted ={pct_over:.1f}%")

uncertain   : median=3314, cutted =2.5%
artifact    : median=3330, cutted =3.1%
real_mid    : median=1366, cutted =0.9%
real_high   : median=1063, cutted =1.4%


we choose MAX_LEN = 8000

In [5]:
NUCLEOTIDE_MAP = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
MAX_LEN = 8000

def one_hot(seq, max_len=MAX_LEN):
    tensor = torch.zeros(4, max_len)
    for i, nuc in enumerate(seq[:max_len]):
        idx = NUCLEOTIDE_MAP.get(nuc, -1)
        if idx >= 0:
            tensor[idx, i] = 1.0
    return tensor

In [6]:
LABEL_MAP = {
    'real_high': 0,
    'real_mid':  1,
    'uncertain': 2,
    'artifact':  3
}

class IsoformDataset(Dataset):
    def __init__(self, sequences, labels, max_len=MAX_LEN):
        self.sequences = sequences
        self.labels = [LABEL_MAP[l] for l in labels]
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = one_hot(self.sequences[idx], self.max_len)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return seq, label

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, 
                                    stratify=df['target'], 
                                    random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1,
                                    stratify=train_df['target'],
                                    random_state=42)

train_ds = IsoformDataset(train_df['sequence'].values, train_df['target'].values)
val_ds   = IsoformDataset(val_df['sequence'].values,   val_df['target'].values)
test_ds  = IsoformDataset(test_df['sequence'].values,  test_df['target'].values)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=4)

print(f"Train: {len(train_ds):,}")
print(f"Val:   {len(val_ds):,}")
print(f"Test:  {len(test_ds):,}")

seq_batch, label_batch = next(iter(train_loader))
print(f"\nseq shape:   {seq_batch.shape}")
print(f"label shape: {label_batch.shape}")

Train: 352,472
Val:   39,164
Test:  97,909

seq shape:   torch.Size([32, 4, 8000])
label shape: torch.Size([32])


In [8]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel = 7, dilation = 1):
        super().__init__()
        padding = (kernel - 1) * dilation // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation, padding=padding)
        self.bn = nn.BatchNorm1d(out_ch)
        self.dp = nn.Dropout(0.1)

    def forward(self, x):
        return self.dp(F.relu(self.bn(self.conv(x))))

In [9]:
class IsoformClassifier(nn.Module):
    def __init__(self, n_classes = 4):
        super().__init__()
        self.conv1 = ConvBlock(4, 128, kernel = 7, dilation = 1)
        self.conv2 = ConvBlock(128, 128, kernel = 7, dilation = 2)
        self.conv3 = ConvBlock(128, 256, kernel = 7, dilation = 4)
        self.conv4 = ConvBlock(256, 256, kernel = 7, dilation = 8)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    
    def forward(self, x):
        x = self.conv4(self.conv3(self.conv2(self.conv1(x))))
        x = x.max(dim = -1).values
        return self.classifier(x)

This model uses Global Max Pooling, which captures whether a pattern 
exists anywhere in the sequence, regardless of its position. 
As a result, the model learns what patterns are present, 
but not where they occur.

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = IsoformClassifier(n_classes=4).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Parameters: {total_params:,}")

with torch.no_grad():
    out = model(seq_batch.to(device))
    print(f"Output shape: {out.shape}")  

Device: cuda
Parameters: 842,116
Output shape: torch.Size([32, 4])


In [11]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(['artifact', 'real_high', 'real_mid', 'uncertain'])
weights = compute_class_weight('balanced', classes = classes, y = df['target'].values)
class_weights = torch.tensor(weights, dtype = torch.float32).to(device)

In [12]:
model = IsoformClassifier(n_classes=4).to(device)

criterion = nn.CrossEntropyLoss(weight = class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer = optimizer, patience = 4, factor = 0.5)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0,0,0
    for seq, label in loader:
        seq, label = seq.to(device), label.to(device)
        out = model(seq)
        loss = criterion(out, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for seq, label in loader:
            seq, label = seq.to(device), label.to(device)
            out = model(seq)
            loss = criterion(out, label)
            total_loss += loss.item()
            correct += (out.argmax(dim=1) == label).sum().item()
            total += label.size(0)
    return total_loss / len(loader), correct / total

In [ ]:
batch_size = 128

from torch.amp import autocast, GradScaler
scaler = GradScaler('cuda')


print(f"Батчей в эпохе: {len(train_loader)}")
print(f"Каждый батч: {batch_size} × 4 × {MAX_LEN} = {batch_size * 4 * MAX_LEN:,} элементов")

import time
model.train()
seq, label = next(iter(train_loader))
seq, label = seq.to(device), label.to(device)

start = time.time()
with autocast('cuda'):
    out = model(seq)
    loss = criterion(out, label)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
torch.cuda.synchronize()
end = time.time()

batch_time = end - start
epoch_time = batch_time * len(train_loader) / 60
print(f"\nВремя одного батча: {batch_time:.2f} сек")
print(f"Время эпохи: {epoch_time:.1f} мин")

Батчей в эпохе: 11015
Каждый батч: 128 × 4 × 8000 = 4,096,000 элементов

Время одного батча: 3.14 сек
Время эпохи: 576.9 мин


In [13]:
EPOCHS = 20
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 
        'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/home/vlad/eQTL/processed/best_model.pt')
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

KeyboardInterrupt: 